# LunarLander PPO Agent

##Install dependencies:

`swig` connects C or C++ scripts with Python, `cmake` is a system generator

In [ ]:
!apt install swig cmake

In [ ]:
!apt-get install ffmpeg freeglut3-dev xvfb  # For visualization
!pip install stable-baselines3[extra]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd drive/MyDrive/ColabNotebooks/DeepRL

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ColabNotebooks/LLMs-from-scratch


## Create a Virtual Screen

In [ ]:
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg
!apt install xvfb
!pip3 install pyvirtualdisplay

Restart the notebook runtime, forcing runtime to crash

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [ ]:
!pip install huggingface-sb3
!pip install stable-baselines3

In [ ]:
!pip install wheel setuptools pip --upgrade

In [ ]:
# install C++ build tools

In [ ]:
# install Box2D
!pip install -q swig
!pip install -q "gymnasium[box2d]"

In [ ]:
!pwd

/content


## Package Import

In [ ]:
import gymnasium

from huggingface_sb3 import load_from_hub, package_to_hub
from huggingface_hub import notebook_login # be able to upload models to the Hub

from stable_baselines3 import PPO
from stable_baselines3.ppo.policies import MlpPolicy
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Gymnasium

1. Create a environment with `gymnasium.make()`
2. Reset the environment to its inital state with `observation = env.reset()`
3. At each step get an action using the model
4. Using `env.step(action)` perform the action in the environment and get
  * `observation`: the new state
  * `reward`
  * `terminated`: indicates if the agent reach the terminal state
  * `truncate`: indicates a timelimit or out of bounds
  * `info`: a dict providing additional info

the environment is reset to its inital state with `observation = env.reset()`

## Create an Environment

In [ ]:
import gymnasium as gym

# create environment called LunarLander-v3
env = gym.make("LunarLander-v3")

# reset this environment
observation, info = env.reset()

for _ in range(20):
  # take a random action
  action = env.action_space.sample()
  print("Action taken:", action)

  # perform this action in the environment and get
  # next_state, reward, terminated, truncated and info
  observation, reward, terminated, truncated, info = env.step(action)

  # if the game is terminated, lunar lander lands, crashes or timeout
  if terminated or truncated:
      # reset the environment
      print("Environment is reset")
      observation, info = env.reset()

env.close()

Action taken: 2
Action taken: 0
Action taken: 3
Action taken: 2
Action taken: 0
Action taken: 2
Action taken: 1
Action taken: 0
Action taken: 2
Action taken: 3
Action taken: 3
Action taken: 3
Action taken: 3
Action taken: 3
Action taken: 1
Action taken: 0
Action taken: 2
Action taken: 2
Action taken: 0
Action taken: 1


### Environment looks like:

**Observation Space** Shape (8, ) that the observation is a vector of size 8, where each value contains different information about the lander:

* Horizontal pad coordinate (x)
* Vertical pad coordinate (y)
* Horizontal speed (x)
* Vertical speed (y)
* Angle
* Angular speed
* If the left leg contact point has touched the land (boolean)
* If the right leg contact point has touched the land (boolean)

**Action space** (the set of possible actions the agent can take) is discrete with 4 actions available 🎮:

* Action 0: Do nothing,
* Action 1: Fire left orientation engine,
* Action 2: Fire the main engine,
* Action 3: Fire right orientation engine.

**Reward function** is granted after every step. The total reward of an episode is the sum of the rewards for all the steps within that episode.

For each step, the reward:

* Is increased/decreased the closer/further the lander is to the landing pad.
* Is increased/decreased the slower/faster the lander is moving.
* Is decreased the more the lander is tilted (angle not horizontal).
* Is increased by 10 points for each leg that is in contact with the ground.
* Is decreased by 0.03 points each frame a side engine is firing.
* Is decreased by 0.3 points each frame the main engine is firing.
The **episode** receive an additional reward of -100 or +100 points for crashing or landing safely respectively.

An episode is considered a solution if it scores at least 200 points.

In [ ]:
env = gym.make("LunarLander-v3")
env.reset()

print("_____OBSERVATION SPACE_____ \n")
print("Observation Space Shape", env.observation_space.shape)
print("Sample observation", env.observation_space.sample()) # get a random observation
print()
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample()) # Take a random action

_____OBSERVATION SPACE_____ 

Observation Space Shape (8,)
Sample observation [-1.6170003  -1.6457542  -6.2816486   4.726482    0.33844838 -7.7191916
  0.66255325  0.38983655]


 _____ACTION SPACE_____ 

Action Space Shape 4
Action Space Sample 3


## Create LunarLander environment

A vectorized environment, a method for stacking multiple independent environments into a single environment of n environments, which allows for agent training on n environments per step, actions, observations, rewards and end of episodes signals (dones) passesed to the environment are a n dim vector

In [ ]:
env = make_vec_env('LunarLander-v3', n_envs=16)

## Create the model

Use **Proximal Policy Optimization** deep reinforcement learning algorithm from the Stable Baselines **SB3** library to build the algorithm to solve the problem: being able to land the Lunar Lander to the Landing Pad by controlling left, right and main orientation engine. PPO is combination of *Value-based Reinforcement Learning* of an action-value function, and *Policy-based Reinforcement Learning* of a policy that will give a probability distribution over actions.

In [ ]:
model = PPO(                             # training parameters
    policy = 'MlpPolicy',
    env = env,
    n_steps = 1024,
    batch_size = 64,
    n_epochs = 4,
    gamma = 0.999,
    gae_lambda = 0.98,
    ent_coef = 0.01,
    verbose=1)

Using cuda device


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Train the PPO Agent

In [ ]:
model_name = "ppo-LunarLander-v3"
model.learn(total_timesteps=int(1e6)) # 1*10^6=1,000,000
model.save(model_name)

## Evaluate the Agent

In [ ]:
# create a new environment, wrapped in Monitor for evaluation
eval_env = Monitor(gym.make("LunarLander-v3", render_mode='rgb_array'))
# evaluate the model with 10 evaluation episodes and deterministic=True
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

mean_reward=239.85 +/- 15.47954051889197


## Publish trained model on HF Hub

In [ ]:
notebook_login()
!git config --global credential.helper store

In [ ]:
# repo_id = {organization}/{repo_name}
repo_id = "elliemci/ppo-LunarLander-v3"
# define name of the environment
env_id = "LunarLander-v3"

# create the evaluation env and set the render_mode="rgb_array"
eval_env = DummyVecEnv([lambda: gym.make(env_id, render_mode="rgb_array")])

# define model architecture
model_architecture = "PPO"
# commit message
commit_message = "PPO LunarLander-v3 trained agent with video_recorder"

# method save, evaluate, generate a model card and record a replay video of your agent before pushing the repo to the hub
package_to_hub(model = model,                           # trained model
               model_name = model_name,                 # trained model name
               model_architecture = model_architecture, # model architecture we used
               env_id = env_id,                         # environment name
               eval_env = eval_env,                     # evaluation environment
               repo_id = repo_id,                       # id of the model repository from the Hugging Face Hub
               commit_message=commit_message)

## Prepare video recording

In [ ]:
import os

os.system("Xvfb :1 -screen 0 1024x768x24 &")
os.environ['DISPLAY'] = ':1'

In [ ]:
import base64

from pathlib import Path
from IPython import display as ipythondisplay


def show_videos(video_path="", prefix=""):
    """
    :param video_path: (str) Path to the folder containing videos
    :param prefix: (str) Filter the video, showing only the only starting with this prefix
    """
    html = []
    for mp4 in Path(video_path).glob("{}*.mp4".format(prefix)):
        video_b64 = base64.b64encode(mp4.read_bytes())
        html.append(
            """<video alt="{}" autoplay
                    loop controls style="height: 400px;">
                    <source src="data:video/mp4;base64,{}" type="video/mp4" />
                </video>""".format(
                mp4, video_b64.decode("ascii")
            )
        )
    ipythondisplay.display(ipythondisplay.HTML(data="<br>".join(html)))

Record a video using VecVideoRecorder wrapper

In [ ]:
from stable_baselines3.common.vec_env import VecVideoRecorder, DummyVecEnv


def record_video(env_id, model, video_length=500, prefix="", video_folder="videos/"):
    """
    :param env_id: (str)
    :param model: (RL model)
    :param video_length: (int)
    :param prefix: (str)
    :param video_folder: (str)
    """
    eval_env = DummyVecEnv([lambda: gym.make(env_id, render_mode="rgb_array")])
    # start the video at step=0 and record 500 steps
    eval_env = VecVideoRecorder(
        eval_env,
        video_folder=video_folder,
        record_video_trigger=lambda step: step == 0,
        video_length=video_length,
        name_prefix=prefix,
    )

    obs = eval_env.reset()
    for _ in range(video_length):
        action, _ = model.predict(obs)
        obs, _, _, _ = eval_env.step(action)

    # close the video recorder
    eval_env.close()

## Visualized trained agent

In [ ]:
record_video("LunarLander-v3", model, video_length=500, prefix="ppo-lunarlander")

Saving video to /content/videos/ppo-lunarlander-step-0-to-step-500.mp4
Moviepy - Building video /content/videos/ppo-lunarlander-step-0-to-step-500.mp4.
Moviepy - Writing video /content/videos/ppo-lunarlander-step-0-to-step-500.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/ppo-lunarlander-step-0-to-step-500.mp4


In [ ]:
show_videos("videos", prefix="ppo")

## Load a saved LunarLander model from the Hub

In [ ]:
import gymnasium as gym

from stable_baselines3 import PPO
from huggingface_sb3 import load_from_hub
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy

repo_id = "elliemci/ppo-LunarLander-v3" # repo_id
filename = "ppo-LunarLander-v3.zip" # the model filename.zip

In [ ]:
custom_objects = {
            "learning_rate": 0.0,
            "lr_schedule": lambda _: 0.0,
            "clip_range": lambda _: 0.0,
}

checkpoint = load_from_hub(repo_id, filename)
model = PPO.load(checkpoint, custom_objects=custom_objects, print_system_info=True)

== CURRENT SYSTEM INFO ==
- OS: Linux-6.1.123+-x86_64-with-glibc2.35 # 1 SMP PREEMPT_DYNAMIC Sun Mar 30 16:01:29 UTC 2025
- Python: 3.12.11
- Stable-Baselines3: 2.7.0
- PyTorch: 2.8.0+cu126
- GPU Enabled: False
- Numpy: 2.0.2
- Cloudpickle: 3.1.1
- Gymnasium: 1.2.0
- OpenAI Gym: 0.25.2

== SAVED MODEL SYSTEM INFO ==
- OS: Linux-6.1.123+-x86_64-with-glibc2.35 # 1 SMP PREEMPT_DYNAMIC Sun Mar 30 16:01:29 UTC 2025
- Python: 3.12.11
- Stable-Baselines3: 2.7.0
- PyTorch: 2.8.0+cu126
- GPU Enabled: True
- Numpy: 2.0.2
- Cloudpickle: 3.1.1
- Gymnasium: 1.2.0
- OpenAI Gym: 0.25.2



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# evaluate the agent
eval_env = Monitor(gym.make("LunarLander-v3"))
mean_reward, std_reward = evaluate_policy(model, eval_env, n_eval_episodes=10, deterministic=True)
print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

mean_reward=254.46 +/- 18.402364015729084
